# Trail Surface ID — YOLO26 mask + CLIP open-vocab

Uses our **YOLO26 instance-segmentation** model to produce a precise trail mask, then runs **CLIP** on the *masked region only* to identify the surface type with an open-vocabulary text prompt.

Why combine them:
- YOLO is great at finding **where** the trail is, even when the surface is something it wasn't trained on.
- CLIP is great at putting an open-vocab label on a region — adding a new surface type means adding a text prompt, no retraining.
- Masking out the non-trail pixels before CLIP keeps surrounding foliage / sky out of the embedding.

## Step 1: Install & import

In [ ]:
!pip install -q ultralytics transformers pillow

In [ ]:
import os
import time
import numpy as np
import torch
import cv2
from PIL import Image
from IPython.display import display
from google.colab import files
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Step 2: Upload YOLO26 weights & load both models

In [ ]:
if 'yolo_model' not in locals():
    print("Upload YOLO26 .pt weights (instance segmentation model):")
    yolo_upload = files.upload()
    yolo_weights_path = next(k for k in yolo_upload if k.endswith('.pt'))
    print(f"\nLoading YOLO from {yolo_weights_path}...")
    yolo_model = YOLO(yolo_weights_path)
    print(f"YOLO classes: {yolo_model.names}")

if 'clip_model' not in locals():
    print("\nLoading CLIP...")
    clip_model_name = "openai/clip-vit-base-patch32"
    clip_model = CLIPModel.from_pretrained(clip_model_name).to(device)
    clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
    print("CLIP loaded.")

## Step 3: Define surface labels for CLIP

Each key is the final label. The list of phrases per key is averaged with a max-vote, so adding more phrasings for a category usually makes it more robust.

Easy to extend — drop in a new key (e.g. `"Sand"`) with a couple of `"a photo of ..."` prompts and rerun.

In [ ]:
surface_categories = {
    "Gravel": [
        "a photo of a gravel trail",
        "a photo of a trail made of loose, small crushed rocks",
        "a photo of a crushed stone path"
    ],
    "Paved": [
        "a photo of a paved asphalt path",
        "a photo of a smooth concrete trail",
        "an artificial sidewalk with clean, straight edges",
        "a smooth surface designed for bicycle tires and car wheels"
    ],
    "Boardwalk": [
        "a photo of a wide wooden boardwalk",
        "a photo of a flat, accessible path built with multiple wooden planks",
        "a photo of a wooden bridge wide enough for a wheelchair"
    ],
    "Bog Bridge": [
        "a photo of a narrow bog bridge",
        "a photo of a single wooden log placed on the trail to walk on",
        "a photo of a narrow wooden plank over mud, difficult for a wheelchair"
    ],
    "Rocky": [
        "a photo of a rocky hiking trail",
        "a photo of uneven ground covered in large boulders and stones"
    ],
    "Dirt": [
        "a photo of a packed dirt trail",
        "a photo of a soil footpath through the woods",
        "a photo of a smooth earthen path"
    ],
    "Woodchips": [
        "a photo of woodchips on a trail",
        "a photo of a path covered in shredded bark and mulch"
    ],
    "Grass": [
        "a photo of a grassy footpath",
        "a photo of a mown grass trail through a field"
    ]
}

# Flatten into a single phrase list for CLIP, with reverse lookup
all_phrases = []
phrase_to_category = {}
for category, phrases in surface_categories.items():
    for phrase in phrases:
        all_phrases.append(phrase)
        phrase_to_category[phrase] = category

print(f"{len(surface_categories)} categories, {len(all_phrases)} prompts.")

## Step 4: Helper functions

In [ ]:
# YOLO classes that represent the trail surface itself.
# Anything matching these (case-insensitive substring match) will be considered a candidate trail mask.
TRAIL_CLASS_HINTS = ["trail", "path", "sidewalk", "road"]

# Confidence floor for a YOLO mask to be considered usable for CLIP
YOLO_TRAIL_MIN_CONF = 0.25


def is_trail_class(class_name):
    cn = class_name.lower()
    return any(h in cn for h in TRAIL_CLASS_HINTS)


def get_best_trail_mask(yolo_result):
    """From a single YOLO Results object, pick the trail-class instance with the largest
    (mask_area * confidence) and return (mask_bool, yolo_class_name, yolo_conf, mask_area_px).
    Returns (None, None, None, 0) if nothing qualifies."""
    if yolo_result.masks is None or yolo_result.boxes is None or len(yolo_result.boxes) == 0:
        return None, None, None, 0

    masks_np = yolo_result.masks.data.cpu().numpy().astype(bool)  # (N, H, W) at model res
    confs    = yolo_result.boxes.conf.cpu().numpy()
    clss     = yolo_result.boxes.cls.cpu().numpy().astype(int)
    names    = yolo_result.names  # dict {id: name}

    best = None
    for i in range(len(confs)):
        cname = names.get(int(clss[i]), str(int(clss[i])))
        if not is_trail_class(cname) or confs[i] < YOLO_TRAIL_MIN_CONF:
            continue
        area = int(masks_np[i].sum())
        score = area * float(confs[i])
        if best is None or score > best[0]:
            best = (score, masks_np[i], cname, float(confs[i]), area)

    if best is None:
        return None, None, None, 0
    _, mask, cname, conf, area = best
    return mask, cname, conf, area


def masked_crop_for_clip(pil_img, mask_bool):
    """Apply the boolean mask to the image (non-trail pixels -> black) and crop to
    the mask's bounding box. Returns a PIL.Image suitable for CLIP."""
    img_np = np.array(pil_img)  # (H, W, 3)
    H, W = img_np.shape[:2]

    # YOLO mask may be at a different resolution than the original image -> resize
    if mask_bool.shape != (H, W):
        mask_resized = cv2.resize(mask_bool.astype(np.uint8), (W, H),
                                  interpolation=cv2.INTER_NEAREST).astype(bool)
    else:
        mask_resized = mask_bool

    masked = img_np.copy()
    masked[~mask_resized] = 0  # black out non-trail pixels

    ys, xs = np.where(mask_resized)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max())
    y1, y2 = int(ys.min()), int(ys.max())
    crop = masked[y1:y2 + 1, x1:x2 + 1]
    return Image.fromarray(crop)


def fallback_bottom_center_crop(pil_img):
    """When YOLO finds no trail mask: use the previous bottom-center crop heuristic
    (left 20%, top 60%, right 80%, bottom 100%) so we still get a CLIP answer."""
    w, h = pil_img.size
    return pil_img.crop((int(w * 0.2), int(h * 0.6), int(w * 0.8), h))


def clip_classify_surface(pil_img):
    """Run CLIP on a PIL image and return (predicted_label, confidence_pct, full_scores_dict)."""
    inputs = clip_processor(text=all_phrases, images=pil_img,
                            return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy()[0]

    category_scores = {cat: 0.0 for cat in surface_categories}
    for phrase, score in zip(all_phrases, probs):
        cat = phrase_to_category[phrase]
        if score > category_scores[cat]:
            category_scores[cat] = float(score)

    predicted = max(category_scores, key=category_scores.get)
    return predicted, category_scores[predicted] * 100, category_scores


def overlay_mask(pil_img, mask_bool, color=(0, 255, 128), alpha=0.45):
    """Return a PIL image with the mask drawn as a semi-transparent overlay (for display)."""
    img_np = np.array(pil_img).copy()
    H, W = img_np.shape[:2]
    if mask_bool.shape != (H, W):
        mask_bool = cv2.resize(mask_bool.astype(np.uint8), (W, H),
                               interpolation=cv2.INTER_NEAREST).astype(bool)
    overlay = img_np.copy()
    overlay[mask_bool] = (alpha * np.array(color) + (1 - alpha) * overlay[mask_bool]).astype(np.uint8)
    return Image.fromarray(overlay)

## Step 5: Upload images

In [ ]:
print("Select trail photos to process...")
uploaded_images = files.upload()
print(f"\n{len(uploaded_images)} image(s) uploaded.")

## Step 6: Main processing loop

For each image:
1. Run YOLO26 instance segmentation
2. Pick the best trail-class mask (largest area × confidence)
3. Apply mask + crop to bbox → CLIP
4. Compare YOLO's class label vs CLIP's open-vocab surface label
5. Fall back to bottom-center crop if YOLO finds no trail

In [ ]:
summary_rows = []

for image_filename in uploaded_images.keys():
    start_time = time.time()
    print(f"\n{'='*60}\nAnalyzing: {image_filename}\n{'='*60}")

    pil_img = Image.open(image_filename).convert("RGB")

    # ---- 1. YOLO segmentation ----
    yolo_results = yolo_model.predict(np.array(pil_img), verbose=False)[0]
    mask, yolo_cls, yolo_conf, mask_area = get_best_trail_mask(yolo_results)

    # ---- 2. Pick CLIP input region ----
    if mask is not None:
        clip_input = masked_crop_for_clip(pil_img, mask)
        source = f"YOLO mask ({yolo_cls}, conf={yolo_conf:.2f}, area={mask_area}px)"
    else:
        clip_input = fallback_bottom_center_crop(pil_img)
        source = "FALLBACK bottom-center crop (YOLO found no trail)"

    # ---- 3. CLIP classification ----
    clip_label, clip_conf, all_scores = clip_classify_surface(clip_input)

    # ---- 4. Display ----
    print(f"CLIP input source: {source}")
    if mask is not None:
        print("\nOriginal with YOLO trail mask overlay:")
        overlay = overlay_mask(pil_img, mask)
        overlay.thumbnail((800, 800))
        display(overlay)

    print("\nRegion CLIP saw (masked + cropped):")
    show = clip_input.copy()
    show.thumbnail((400, 400))
    display(show)

    # ---- 5. Results ----
    print(f"\nYOLO surface class:  {yolo_cls or 'no trail mask'}"
          f"{f' ({yolo_conf*100:.1f}%)' if yolo_conf is not None else ''}")
    print(f"CLIP surface class:  {clip_label.upper()} ({clip_conf:.1f}%)")

    # Flag disagreement so it's easy to spot images outside YOLO's trained classes
    yolo_norm = (yolo_cls or '').lower().replace(' trail', '').replace(' ', '')
    clip_norm = clip_label.lower().replace(' ', '')
    if yolo_cls and yolo_norm not in clip_norm and clip_norm not in yolo_norm:
        print("  -> YOLO and CLIP disagree (CLIP likely captured an out-of-vocab surface)")

    print("\nCLIP leaderboard:")
    for cat, score in sorted(all_scores.items(), key=lambda x: -x[1]):
        bar = "█" * int(score * 30)
        print(f"  {cat:12s} {score*100:5.1f}%  {bar}")

    print(f"\n(Processed in {time.time() - start_time:.2f}s)")

    summary_rows.append({
        "image": image_filename,
        "yolo_class": yolo_cls,
        "yolo_conf": round(yolo_conf, 3) if yolo_conf is not None else None,
        "yolo_mask_area_px": mask_area,
        "clip_label": clip_label,
        "clip_conf_pct": round(clip_conf, 1),
        "used_fallback": mask is None,
    })

## Step 7: Summary table

In [ ]:
import pandas as pd
summary_df = pd.DataFrame(summary_rows)
summary_df